# Week 10 Solution Notebook

This notebook solves the questions based on `Resources/Data/Week-10-GA-1.json`.
It converts the JSON data into a one-hot encoded Pandas DataFrame where:

- rows are movies
- columns are genres
- each cell is `1` if the movie belongs to that genre, else `0`


In [ ]:
from pathlib import Path
import json
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity


## Load the JSON data


In [ ]:
data_path = Path("Resources/Data/Week-10-GA-1.json")
with data_path.open("r", encoding="utf-8") as f:
    movie_genres = json.load(f)

len(movie_genres), list(movie_genres.items())[:3]


## Create the movie-genre DataFrame


In [ ]:
all_genres = sorted({genre for genres in movie_genres.values() for genre in genres})
movies = sorted(movie_genres.keys(), key=lambda name: int(name.split()[1]))

genre_df = pd.DataFrame(0, index=movies, columns=all_genres, dtype=int)

for movie, genres in movie_genres.items():
    genre_df.loc[movie, genres] = 1

genre_df.head()


## 1. Unique number of genres present in the dataset


In [ ]:
num_unique_genres = genre_df.shape[1]
num_unique_genres


## 2. Cosine similarity between `Movie 1` and `Movie 10`


In [ ]:
movie_1_vs_10 = cosine_similarity(
    genre_df.loc[["Movie 1"]],
    genre_df.loc[["Movie 10"]]
)[0, 0]

movie_1_vs_10


## 3. Movies most similar to `Movie 50`


In [ ]:
movie_50_scores = pd.Series(
    cosine_similarity(genre_df.loc[["Movie 50"]], genre_df)[0],
    index=genre_df.index,
    name="cosine_similarity"
)

movie_50_scores = movie_50_scores.drop("Movie 50")
movie_50_scores.sort_values(ascending=False).head(10)


In [ ]:
top_5_similar = movie_50_scores.sort_values(ascending=False).head(5)
top_5_similar.index.tolist()


## Final answers

1. Unique number of genres: **15**
2. Cosine similarity between `Movie 1` and `Movie 10`: **0.3162277660**
3. The similarity ranking for `Movie 50` starts with:

- `Movie 72` with score `0.75`
- `Movie 1` with score `0.6708203932`
- `Movie 2`, `Movie 8`, `Movie 61`, and `Movie 76` are tied with score `0.5773502692`

Because of this tie, the top-5 list is not strictly unique. If your quiz expects one of the given choices, **option c** is the closest match, but it omits the tied movie `Movie 8`.


# Week 10 GA-2: User-User Collaborative Filtering

This section uses `Resources/Data/Week-10-GA-2.csv` to build a user-item matrix, compute the user-user cosine similarity matrix, identify the most similar pair among the listed options, and predict the rating of item `J` for `User 1`.


In [ ]:
ratings_path = Path("Resources/Data/Week-10-GA-2.csv")
ratings_df = pd.read_csv(ratings_path)

# In this file, columns are users and rows are items A-J.
ratings_df.index = list("ABCDEFGHIJ")
ratings_df


## Build the user-item matrix


In [ ]:
user_item_df = ratings_df.T
user_item_df


## User-User cosine similarity matrix


In [ ]:
user_similarity = pd.DataFrame(
    cosine_similarity(user_item_df),
    index=user_item_df.index,
    columns=user_item_df.index,
)

user_similarity


## Which listed pair has the highest similarity?


In [ ]:
candidate_pairs = {
    "a": ("User 8", "User 10"),
    "b": ("User 1", "User 9"),
    "c": ("User 2", "User 5"),
    "d": ("User 7", "User 3"),
}

pair_scores = pd.Series(
    {label: user_similarity.loc[u1, u2] for label, (u1, u2) in candidate_pairs.items()},
    name="similarity_score",
).sort_values(ascending=False)

pair_scores


## Predict the rating of item `J` for `User 1`

We use standard user-user collaborative filtering with a weighted average of other users' ratings for item `J`:

$$\hat{r}_{u,i} = \frac{\sum_{v \ne u} s(u,v) \cdot r_{v,i}}{\sum_{v \ne u} |s(u,v)|}$$


In [ ]:
target_user = "User 1"
target_item = "J"

similar_users = user_similarity.loc[target_user].drop(target_user)
item_j_ratings = user_item_df[target_item].drop(target_user)

prediction_user1_j = (similar_users * item_j_ratings).sum() / similar_users.abs().sum()
prediction_user1_j


## Final answers for GA-2

1. Highest similarity among the listed pairs: **option d (`User 7` and `User 3`)**
2. Predicted rating of item `J` for `User 1`: **3.0742166807**

If your course rounds the predicted score to the nearest integer, the rating is **3**.
